<a href="https://colab.research.google.com/github/amilynestes1028/ds2002-fa26/blob/main/notebooks/01-foundations/2026_09_11_%E2%80%94_SQL_Challenge_Set_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [17]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [18]:
q1 = q('''
SELECT
t.track_id,
t.title,
t.genre,
t.seconds,
a.name AS artist_name,
a.country AS artist_country
FROM tracks t
LEFT JOIN artists a ON t.artist_id = a.artist_id
''')

In [19]:
display(q1)

,track_id,title,genre,seconds,artist_name,artist_country
0,10,Skyline,Pop,201,Nova Waves,US
1,11,Undertow,Pop,240,Nova Waves,US
2,12,Foothills,Folk,185,The Blue Ridge,US
3,13,Aurora,Electronic,300,Kestrel,UK
4,14,Nightfall,Electronic,275,Kestrel,UK
5,15,Sol,Latin,210,Marisol,ES
6,16,Coastline,Folk,199,The Blue Ridge,US
7,17,Ridgeline,Folk,225,The Blue Ridge,US
8,18,Untitled Demo,None,150,Kestrel,UK


### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [20]:
q2 = q('''
SELECT
    COALESCE(genre, 'Untagged') AS genre,
    AVG(seconds) AS avg_length_seconds
FROM tracks
GROUP BY COALESCE(genre, 'Untagged')
ORDER BY avg_length_seconds DESC
LIMIT 1;
''')

In [21]:
display(q2)

,genre,avg_length_seconds
0,Electronic,287.5


### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [22]:
q3 = q('''
SELECT
    user,
    COUNT(play_id) AS plays,
    COUNT(DISTINCT track_id) AS distinct_tracks
FROM plays
GROUP BY user
ORDER BY user;
''')

In [23]:
display(q3)

,user,plays,distinct_tracks
0,ava,4,4
1,ben,3,3
2,cara,2,2
3,dan,2,2


### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [24]:
q4 = q('''
SELECT
    t.track_id,
    t.title,
    t.genre,
    a.name AS artist_name
FROM tracks t
LEFT JOIN artists a ON t.artist_id = a.artist_id
LEFT JOIN plays p ON t.track_id = p.track_id
WHERE p.play_id IS NULL;
''')

In [25]:
display(q4)

,track_id,title,genre,artist_name
0,17,Ridgeline,Folk,The Blue Ridge
1,18,Untitled Demo,None,Kestrel


### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [26]:
q5 = q('''
SELECT
    a.name AS artist,
    COALESCE(SUM(t.seconds), 0) AS total_seconds,
    ROUND(COALESCE(SUM(t.seconds), 0) / 60.0, 1) AS total_minutes
FROM artists a
JOIN tracks t ON a.artist_id = t.artist_id
JOIN plays p ON t.track_id = p.track_id
GROUP BY a.artist_id, a.name
ORDER BY total_seconds DESC;
''')

In [27]:
display(q5)

,artist,total_seconds,total_minutes
0,Kestrel,1175,19.6
1,Nova Waves,843,14.1
2,The Blue Ridge,384,6.4
3,Marisol,210,3.5


### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [28]:
q6 = q('''
SELECT
    track_id,
    title
FROM tracks
WHERE genre IS NULL;
''')

In [29]:
display(q6)

,track_id,title
0,18,Untitled Demo


### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [30]:
q7 = q('''
SELECT
    played_on AS play_date,
    COUNT(play_id) AS total_plays,
    COUNT(DISTINCT user) AS active_users
FROM plays
GROUP BY played_on
ORDER BY played_on ASC;
''')

In [31]:
display(q7)

,play_date,total_plays,active_users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [32]:
assert len(q1) == 9, 'Q1 should return one row per track'
assert len(q4) == 2, 'Q4: two tracks have never been played'
assert q3['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

What caused me the most confusion and gave me the biggest issue was towards the end when i was supposed to assign my query results to the variables. I didnt intially do that as the instructions said, so then when I had to valdiate my work I was super confused why I was getting multiple error statements. I misunderstood the meaning of assigning results to the variables. As well, once I did it, I couldn't see the displayed results anymore so to help that, I added the extra line of code after to display the results of each question.